# Image Captioning — Vision-Language Understanding with BLIP-2

Describing an image in natural language requires bridging two fundamentally different
modalities: a grid of pixel values and a sequence of words. This notebook shows how
BLIP-2 solves this bridge problem with a three-stage architecture — a frozen vision
encoder, a lightweight Q-Former adapter, and a pretrained language model — without
retraining any of the large components.

| Step | Concept | Key Idea |
|------|---------|---------|
| 1 | Why image captioning is hard | Color histograms cannot describe composition or semantics |
| 2 | BLIP-2 architecture | Vision encoder + Q-Former + LLM; only Q-Former is trained |
| 3 | Pipeline walkthrough | Processor encodes image; model generates caption tokens |
| 4 | Live demo | Caption a synthetic image; observe prompt-guided output |
| 5 | Your turn | Change max_length or add a text prompt |
| Summary | Key insights | Consolidated takeaways |


In [ ]:
# ── Install dependencies (skip if already present) ─────────────────────────────
import importlib, subprocess, sys

def _ensure(pkg, import_name=None):
    name = import_name or pkg
    try:
        importlib.import_module(name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "--quiet"])

_ensure("transformers")
_ensure("torch")
_ensure("Pillow", "PIL")
_ensure("matplotlib")
_ensure("numpy")
print("Dependencies ready.")


In [ ]:
# ── Imports and deterministic seed ─────────────────────────────────────────────
import torch
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Torch version : {torch.__version__}")
print(f"Device        : {'cuda' if torch.cuda.is_available() else 'cpu'}")


---

## Part 1 — Why does image captioning resist simple rules?

Before vision-language models existed, the best automated approach was to extract
hand-crafted image features (color histograms, edge detectors, texture descriptors)
and look them up in a template library. The method fails on composition:
it can detect "blue" and "sky" and "bird" as separate features but cannot produce
"a white bird soaring above a bright blue lake."

Predict before you run:
> A color histogram of an image with a red car on a grey road will contain lots of
> red and grey pixels. If you rank the words "car", "road", "red", "drive", "traffic"
> by how easy they are to recover from a color histogram alone — in what order would
> you rank them?

Run the cell below to see what a histogram-only approach actually recovers.


In [ ]:
# ── Naive feature extraction: color histogram ───────────────────────────────────
# We build a synthetic "red car on grey road" image and extract a histogram.

# Create a synthetic scene: top half = grey road, red rectangle = car
img_array = np.ones((200, 300, 3), dtype=np.uint8) * 128   # grey background
img_array[80:160, 100:200, :] = [200, 30, 30]              # red car rectangle
img_array[160:, :, :] = [80, 80, 80]                       # darker road below

synthetic_img = Image.fromarray(img_array)

# Compute a coarse RGB histogram (8 bins per channel)
hist_r, _ = np.histogram(img_array[:, :, 0].ravel(), bins=8, range=(0, 255))
hist_g, _ = np.histogram(img_array[:, :, 1].ravel(), bins=8, range=(0, 255))
hist_b, _ = np.histogram(img_array[:, :, 2].ravel(), bins=8, range=(0, 255))

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].imshow(img_array)
axes[0].set_title("Synthetic scene: red car on grey road")
axes[0].axis("off")

x = np.arange(8)
axes[1].bar(x - 0.25, hist_r, width=0.25, color="red",   label="R channel")
axes[1].bar(x,         hist_g, width=0.25, color="green", label="G channel")
axes[1].bar(x + 0.25, hist_b, width=0.25, color="blue",  label="B channel")
axes[1].set_title("Color histogram (8 bins per channel)")
axes[1].set_xlabel("Bin")
axes[1].legend()
plt.tight_layout()
plt.show()

print("Dominant pixel ranges:")
print(f"  R bins with most pixels: {list(np.argsort(hist_r)[-3:][::-1])}")
print(f"  G bins with most pixels: {list(np.argsort(hist_g)[-3:][::-1])}")
print(f"  B bins with most pixels: {list(np.argsort(hist_b)[-3:][::-1])}")
print()
print("  -> Histogram tells us about colour distribution, not semantics.")
print("  -> 'car', 'road', 'driving' are unrecoverable from pixel counts alone.")


#### What just happened — and what's missing

The histogram captures which colours dominate but nothing about their arrangement.
It cannot distinguish a red car on grey road from a red sunset over grey mountains —
the colour distributions could be identical.

Compositionality — the fact that meaning arises from *where* things are relative to
each other, not just *what* colours appear — requires a model that processes spatial
structure. That is what a vision encoder does.


---

## Part 2 — BLIP-2: three frozen giants, one trainable bridge

BLIP-2 (Bootstrapping Language-Image Pre-training 2) avoids training large components
from scratch by freezing a pretrained vision encoder and a pretrained LLM, and training
only a small connector called the **Q-Former** between them.

**Q-Former query mechanism:**

$$Z = \text{CrossAttention}(Q_{\text{learnable}},\, K,V = E_{\text{image}})$$

Plain-English gloss: a fixed set of $N=32$ learnable query vectors attend to the image
encoder's patch embeddings via cross-attention. The result $Z$ is a compressed
representation that the frozen LLM can understand — bridging the vision and language
embedding spaces with minimal new parameters.

| Stage | Component | Trainable? | Role |
|-------|-----------|-----------|------|
| 1 | ViT-G/14 (EVA-CLIP) | Frozen | Extract patch-level image features |
| 2 | Q-Former (BERT-like) | Trained | Bridge image patches to LLM input space |
| 3 | OPT-2.7B or FlanT5-XL | Frozen | Generate text from Q-Former output |

The Q-Former has only ~188 M parameters, versus ~2.7 B for the LLM and ~1 B for ViT-G.
Training budget is concentrated where it is most needed: the modality bridge.


In [ ]:
# ── Illustrate the Q-Former query-attention concept (toy version) ──────────────
# We build a miniature cross-attention to show how 32 learnable queries
# compress 196 image patches into a fixed-size representation.

import torch
import torch.nn.functional as F

torch.manual_seed(SEED)

N_PATCHES  = 196   # ViT-G/14 with 14x14 patches on a 224x224 image
D_PATCH    = 64    # toy patch embedding dimension (real: 1408)
N_QUERIES  = 32    # Q-Former fixed query count (same as real BLIP-2)
D_QUERY    = 64    # toy query dimension

# Simulated patch embeddings from a frozen vision encoder
patch_embeddings = torch.randn(N_PATCHES, D_PATCH)

# Learnable Q-Former queries
learnable_queries = torch.nn.Parameter(torch.randn(N_QUERIES, D_QUERY))

# Toy linear projections for K, V from patches
W_K = torch.nn.Linear(D_PATCH, D_QUERY, bias=False)
W_V = torch.nn.Linear(D_PATCH, D_QUERY, bias=False)

K = W_K(patch_embeddings)           # (196, 64)
V = W_V(patch_embeddings)           # (196, 64)
Q = learnable_queries                # (32, 64)

# Scaled dot-product attention: Q attends over all 196 patches
scale  = D_QUERY ** 0.5
scores = Q @ K.T / scale            # (32, 196)
attn   = F.softmax(scores, dim=-1)  # (32, 196)
Z      = attn @ V                   # (32, 64)  <- compressed representation

print(f"Input  : {N_PATCHES} image patches  x {D_PATCH}-dim each")
print(f"Queries: {N_QUERIES} learnable vectors x {D_QUERY}-dim each")
print(f"Output : {Z.shape[0]} query outputs x {Z.shape[1]}-dim each")
print()
print("  -> 196 patches compressed to 32 query vectors via cross-attention.")
print("  -> Each query learns to attend to a different aspect of the image.")
print("  -> These 32 vectors are what the frozen LLM receives — not raw pixels.")


#### What just happened — and what's missing

The toy Q-Former compresses 196 patch embeddings into 32 fixed-length vectors by
having each learnable query attend over all patches.
The number 32 is a hyperparameter: fewer queries = more compression but possible
loss of detail; more queries = less compression but more LLM context tokens used.

In the real BLIP-2, the Q-Former also has self-attention between the 32 queries, and
the training objective alternates between image-text matching and image-grounded
text generation — we skipped that complexity to focus on the shape of the computation.


---

## Part 3 — Pipeline walkthrough: from image to caption

`Blip2Processor` handles both the image preprocessing (resize, normalize) and the
optional text tokenization (for prompted captioning). The processor returns a dict
of tensors that feed directly into `Blip2ForConditionalGeneration.generate()`.


In [ ]:
# ── Load BLIP-2 (downloads ~5 GB on first run) ─────────────────────────────────
# blip2-opt-2.7b uses OPT-2.7B as the LLM; blip2-flan-t5-xl uses FlanT5-XL.
# Both share the same ViT-G/14 vision encoder and Q-Former.

from transformers import Blip2Processor, Blip2ForConditionalGeneration

MODEL_NAME = "Salesforce/blip2-opt-2.7b"
print(f"Loading {MODEL_NAME} (vision encoder + Q-Former + OPT-2.7B LLM) ...")

processor = Blip2Processor.from_pretrained(MODEL_NAME, use_fast=False)
blip2     = Blip2ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
blip2.eval()

print("  -> BLIP-2 loaded.")


def caption_image(pil_img: Image.Image,
                  prompt: str = None,
                  max_length: int = 50) -> str:
    """Generate a caption for a PIL image, optionally guided by a text prompt."""
    # Processor handles both image normalization and optional text tokenization
    inputs = processor(images=pil_img, text=prompt, return_tensors="pt")

    # Move tensors to the same device as the model
    device = next(blip2.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = blip2.generate(**inputs, max_length=max_length)

    caption = processor.decode(output_ids[0], skip_special_tokens=True).strip()
    return caption

print("caption_image() ready.")


#### What just happened — and what's missing

`Blip2Processor` unifies image and text preprocessing behind one API:
- For the image: resize to 224x224, normalize with CLIP statistics.
- For the text prompt (optional): tokenize with the LLM's tokenizer.

The `generate()` call runs the full three-stage pipeline internally:
ViT-G encodes the image patches, Q-Former compresses to 32 query vectors,
those vectors are prepended to any text tokens, and OPT-2.7B decodes the caption.


---

## Part 4 — Live demo

We test on a synthetic image (no download needed) and on an optional URL image.


In [ ]:
# ── Live demo: caption the synthetic car-on-road image from Part 1 ─────────────

# Use the synthetic image created in Part 1 (already in memory as synthetic_img)
caption_free   = caption_image(synthetic_img, prompt=None, max_length=50)
caption_guided = caption_image(synthetic_img, prompt="Question: What color is the car? Answer:", max_length=30)

print("Image: synthetic red car on grey road (196x300 px)")
print()
print(f"Free-form caption:")
print(f"  -> {caption_free}")
print()
print(f"Prompted caption (VQA style):")
print(f"  -> {caption_guided}")
print()
print("  -> The prompt prefix shifts the decoder from open captioning to QA mode.")


In [ ]:
# ── Your turn — change max_length or the prompt ────────────────────────────────
# CHANGE: adjust MAX_LENGTH or PROMPT below and re-run.
#    - Shorter max_length -> more concise caption (may truncate)
#    - Longer  max_length -> more descriptive caption
#    - Different prompt   -> steers what the model focuses on

MAX_LENGTH = 30           # try: 10, 80
PROMPT     = None         # try: "A photo of", "Describe the colors:", "Question: What is in the image? Answer:"

result = caption_image(synthetic_img, prompt=PROMPT, max_length=MAX_LENGTH)
print(f"max_length : {MAX_LENGTH}")
print(f"prompt     : {repr(PROMPT)}")
print(f"  -> Caption: {result}")
print("  -> Observe how max_length and prompt independently shape the output.")


---

## Summary

| Step | Concept | Key Idea |
|------|---------|---------|
| 1 | Why captioning is hard | Colour histograms capture distribution, not spatial composition |
| 2 | BLIP-2 architecture | Frozen ViT + trainable Q-Former + frozen LLM; bridge is cheap |
| 3 | Pipeline walkthrough | Processor normalises image; generate() chains all three stages |
| 4 | Live demo | Free-form and prompted captions from the same model |
| 5 | Your turn | max_length and prompt are the two main output-shaping knobs |

**Key insights to keep**

- BLIP-2's key innovation is keeping the large components frozen and training only the Q-Former bridge — this makes vision-language alignment cheap relative to training from scratch.
- The Q-Former compresses a variable number of image patches (196 for 224x224) into exactly 32 query vectors, giving the LLM a fixed-size visual token budget.
- A text prompt prepended before generation shifts the model from open captioning to visual question answering — same weights, different output style.
- Modality bridging (vision -> language) is a general design pattern: freeze large pretrained modules and train only the connector between them.
